# 數據可視化工具

## 學習目標

完成本 Colab 後，你將能夠：

1. 說明數據可視化在大數據分析中的角色。
2. 依資料型態選擇適合的圖表，例如直方圖、箱型圖、散佈圖、熱力圖、長條圖與折線圖。
3. 使用 Python 建立常見的探索性資料視覺化圖表。
4. 解讀圖表中的分佈、異常值、關聯性、類別差異與時間趨勢。
5. 建立簡易儀表板式圖表，用於商業決策溝通。

## 情境設定

你是某線上零售平台的資料分析助理，需要將交易、顧客、商品與時間序列資料轉換成容易理解的圖表，協助主管快速掌握營運狀況。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節會用到的套件，並建立可重複使用的範例資料集。資料模擬線上零售平台的交易紀錄，包含消費金額、停留時間、廣告花費、轉換率、商品類別與月份。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

n = 600
categories = np.random.choice(['3C', '服飾', '食品', '家用品'], size=n, p=[0.28, 0.32, 0.25, 0.15])
regions = np.random.choice(['北部', '中部', '南部', '東部'], size=n, p=[0.42, 0.25, 0.25, 0.08])

ad_spend = np.random.gamma(shape=4, scale=120, size=n)
stay_time = np.random.normal(loc=6, scale=2, size=n).clip(0.5, None)
purchase_amount = 300 + ad_spend * 1.8 + stay_time * 80 + np.random.normal(0, 180, size=n)
purchase_amount = purchase_amount.clip(50, None)
conversion_rate = 0.02 + ad_spend / ad_spend.max() * 0.15 + np.random.normal(0, 0.02, size=n)
conversion_rate = conversion_rate.clip(0.01, 0.35)

sales = pd.DataFrame({
    '商品類別': categories,
    '地區': regions,
    '廣告花費': ad_spend,
    '停留時間': stay_time,
    '消費金額': purchase_amount,
    '轉換率': conversion_rate
})

monthly = pd.DataFrame({
    '月份': pd.date_range('2025-01-01', periods=12, freq='MS'),
    '營收': np.linspace(80000, 150000, 12) + np.random.normal(0, 9000, 12),
    '活躍用戶': np.linspace(12000, 26000, 12) + np.random.normal(0, 1200, 12)
})

print('交易資料筆數:', len(sales))
print('欄位:', list(sales.columns))
display(sales.head())
display(monthly.head())


## 核心概念說明

數據可視化的重點不是把資料畫成漂亮圖表，而是讓使用者更快理解資料特性、發現異常，並支援決策。

常見圖表可依資料問題分成四類：

1. **數值型資料分佈**：用直方圖、箱型圖或 KDE 曲線觀察集中趨勢、偏態與離群值。
2. **變數關聯**：用散佈圖與相關係數矩陣觀察變數間是否存在正相關、負相關或共線性。
3. **類別比較與比例**：用長條圖、堆疊長條圖或圓餅圖比較不同類別的數量、比例與貢獻。
4. **時間序列變化**：用折線圖觀察趨勢、季節性與事件造成的變化。

實務上，圖表選擇應先問三件事：

- 這份資料的主要型態是數值、類別、時間，還是多變數？
- 使用者要比較大小、看趨勢、找異常，還是探索關聯？
- 圖表是否能避免誤導，例如過多類別、過度擁擠、顏色不清或比例難以比較？


In [ ]:
# ── 示範：數值型資料的分佈與異常值 ─────────────────────────
# 這段程式碼示範直方圖與箱型圖。直方圖適合觀察消費金額的整體分佈；箱型圖適合比較不同商品類別的消費金額差異與離群值。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n = 600
categories = np.random.choice(['3C', '服飾', '食品', '家用品'], size=n, p=[0.28, 0.32, 0.25, 0.15])
ad_spend = np.random.gamma(shape=4, scale=120, size=n)
stay_time = np.random.normal(loc=6, scale=2, size=n).clip(0.5, None)
purchase_amount = 300 + ad_spend * 1.8 + stay_time * 80 + np.random.normal(0, 180, size=n)
purchase_amount = purchase_amount.clip(50, None)

sales = pd.DataFrame({'商品類別': categories, '消費金額': purchase_amount})

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(sales['消費金額'], bins=25, color='#4C78A8', edgecolor='white')
plt.title('消費金額直方圖')
plt.xlabel('消費金額')
plt.ylabel('頻次')

plt.subplot(1, 2, 2)
box_data = [sales.loc[sales['商品類別'] == c, '消費金額'] for c in ['3C', '服飾', '食品', '家用品']]
plt.boxplot(box_data, patch_artist=True)
plt.xticks(range(1, 5), ['3C', '服飾', '食品', '家用品'])  # 跨 matplotlib 版本安全：
# boxplot(labels=) 於 3.9 棄用、3.11 移除，改名為 tick_labels；用 xticks 兩邊都能跑
plt.title('不同商品類別的消費金額箱型圖')
plt.xlabel('商品類別')
plt.ylabel('消費金額')

plt.tight_layout()
plt.show()

print('消費金額摘要統計')
display(sales.groupby('商品類別')['消費金額'].describe().round(2))


In [ ]:
# ── 示範：變數關聯與相關係數矩陣 ──────────────────────────
# 這段程式碼示範散佈圖與相關係數熱力圖。散佈圖可觀察廣告花費與轉換率的關聯；相關係數矩陣可快速檢查多個數值變數之間是否高度相關。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
n = 500
ad_spend = np.random.gamma(shape=4, scale=120, size=n)
stay_time = np.random.normal(loc=6, scale=2, size=n).clip(0.5, None)
page_views = stay_time * 2.5 + np.random.normal(0, 2, size=n)
purchase_amount = 300 + ad_spend * 1.7 + stay_time * 90 + np.random.normal(0, 200, size=n)
conversion_rate = 0.02 + ad_spend / ad_spend.max() * 0.16 + stay_time * 0.004 + np.random.normal(0, 0.02, size=n)
conversion_rate = conversion_rate.clip(0.01, 0.35)

sales = pd.DataFrame({
    '廣告花費': ad_spend,
    '停留時間': stay_time,
    '瀏覽頁數': page_views,
    '消費金額': purchase_amount,
    '轉換率': conversion_rate
})

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(sales['廣告花費'], sales['轉換率'], alpha=0.55, color='#F58518')
plt.title('廣告花費與轉換率散佈圖')
plt.xlabel('廣告花費')
plt.ylabel('轉換率')

plt.subplot(1, 2, 2)
corr = sales.corr(numeric_only=True)
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='相關係數')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('相關係數矩陣')

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('與轉換率最相關的變數：')
print(corr['轉換率'].drop('轉換率').sort_values(ascending=False).round(3))


## 如何選擇合適的視覺化工具

在 iPAS 中級考試情境中，常見考點不是記住某個套件的語法，而是判斷「什麼資料問題適合什麼圖表」。

| 分析目的 | 建議圖表 | 常見誤用 |
|---|---|---|
| 看數值分佈 | 直方圖、箱型圖、KDE | bins 選錯導致分佈被扭曲 |
| 比較類別大小 | 長條圖 | 類別太多仍硬畫在同一張圖 |
| 看部分占整體比例 | 圓餅圖、堆疊長條圖 | 類別超過 5 到 6 個仍使用圓餅圖 |
| 看兩變數關聯 | 散佈圖、趨勢線 | 點太多且未設定透明度，造成重疊 |
| 看多變數關係 | 相關係數矩陣、熱力圖 | 把相關解讀成因果 |
| 看時間變化 | 折線圖 | 時間順序未排序，導致趨勢錯誤 |

重要觀念：視覺化是溝通工具。圖表要能支援行動判斷，而不是只展示資料存在。


In [ ]:
# ── 實際應用：建立營運儀表板 ────────────────────────────
# 這段程式碼將類別比較、比例結構與時間趨勢整合成一個簡易營運儀表板，模擬分析師向主管報告商品營收、地區營收與月營收趨勢。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(21)
n = 800
categories = np.random.choice(['3C', '服飾', '食品', '家用品'], size=n, p=[0.30, 0.30, 0.25, 0.15])
regions = np.random.choice(['北部', '中部', '南部', '東部'], size=n, p=[0.42, 0.25, 0.25, 0.08])

unit_price = np.select(
    [categories == '3C', categories == '服飾', categories == '食品', categories == '家用品'],
    [2600, 900, 450, 1200]
)
quantity = np.random.poisson(lam=2.2, size=n) + 1
revenue = unit_price * quantity * np.random.normal(loc=1.0, scale=0.18, size=n)
revenue = revenue.clip(100, None)

sales = pd.DataFrame({
    '商品類別': categories,
    '地區': regions,
    '購買件數': quantity,
    '營收': revenue
})

monthly = pd.DataFrame({
    '月份': pd.date_range('2025-01-01', periods=12, freq='MS'),
    '營收': np.linspace(85000, 165000, 12) + np.random.normal(0, 10000, 12),
    '訂單數': np.linspace(900, 1700, 12) + np.random.normal(0, 100, 12)
})
monthly['營收'] = monthly['營收'].clip(0, None)
monthly['訂單數'] = monthly['訂單數'].round().astype(int).clip(0, None)

category_revenue = sales.groupby('商品類別')['營收'].sum().sort_values(ascending=False)
region_revenue = sales.groupby('地區')['營收'].sum().sort_values(ascending=False)
region_share = region_revenue / region_revenue.sum()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].bar(category_revenue.index, category_revenue.values, color='#4C78A8')
axes[0, 0].set_title('各商品類別營收')
axes[0, 0].set_xlabel('商品類別')
axes[0, 0].set_ylabel('營收')
axes[0, 0].ticklabel_format(axis='y', style='plain')

axes[0, 1].bar(region_revenue.index, region_revenue.values, color='#59A14F')
axes[0, 1].set_title('各地區營收')
axes[0, 1].set_xlabel('地區')
axes[0, 1].set_ylabel('營收')
axes[0, 1].ticklabel_format(axis='y', style='plain')

axes[1, 0].pie(
    region_share.values,
    labels=region_share.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4C78A8', '#F58518', '#59A14F', '#E45756']
)
axes[1, 0].set_title('地區營收占比')

axes[1, 1].plot(monthly['月份'], monthly['營收'], marker='o', color='#F58518', label='營收')
axes[1, 1].set_title('月營收趨勢')
axes[1, 1].set_xlabel('月份')
axes[1, 1].set_ylabel('營收')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].ticklabel_format(axis='y', style='plain')
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print('商品類別營收排名')
display(category_revenue.round(0).astype(int).reset_index(name='營收'))

print('地區營收占比')
display((region_share * 100).round(1).reset_index(name='營收占比(%)'))
